# Vincio: notebook-native analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohswedd/vincio/blob/main/examples/notebooks/06_notebook_native_analysis.ipynb)

The same **governed, cited, offline-verifiable** analysis you run in a script runs
**interactively** here — without becoming a hosted notebook service. Two pieces bind the
data plane into the notebook:

- **rich reprs** — a `QueryResult`, an `AnalysisResult`, a `Chart`, and the sealed
  `DataNarrative` render inline as cards with **clickable cell citations**, the lineage
  verdict, and the audit id;
- **a governed session** — `notebook_session(app, ...)` threads
  register → query → analyze → chart → cite into the **same signed `DataNarrative`** a
  script produces, so the exploration is reproducible by construction and
  `session.verify()` re-derives every inline finding from the bytes.

Everything runs with **no network and no API keys** on the bundled mock provider.

> **To run against a real model**, set `VINCIO_PROVIDER` and the matching API key (e.g.
> `VINCIO_PROVIDER=openai`, `VINCIO_MODEL=gpt-4o-mini`, `OPENAI_API_KEY=...`). The data
> plane is model-free; the provider only powers downstream natural-language answers.

In [ ]:
!pip install -q vincio

## Setup: an offline-first provider and inline reprs

`notebook_session(...)` enables the rich reprs for you (`rich=True`), so cited artifacts
render as cards as soon as a verb returns them.

In [ ]:
import os

import vincio.notebook as nb
from vincio import ContextApp
from vincio.providers import MockProvider, build_provider


def _provider():
    """Offline mock by default; a real provider when VINCIO_PROVIDER is set."""
    name = os.environ.get("VINCIO_PROVIDER", "mock")
    if name == "mock":
        return MockProvider(), "mock-1"
    return build_provider(name), os.environ.get("VINCIO_MODEL", "gpt-4o-mini")


provider, model = _provider()
app = ContextApp(name="analytics", provider=provider, model=model)
nb.enable_rich_reprs()
print(f"provider ready: model={model!r}")

## Open a governed session

A session is a thin, interactive front over `app.data_engagement`. Each verb delegates to
the *same* governed primitive a script calls, captures the artifact, and records it as a
stage in one hash-linked narrative — so a notebook exploration is governed by construction.

In [ ]:
SALES = [
    {"region": "NA", "product": "alpha", "revenue": 1200.5},
    {"region": "EU", "product": "alpha", "revenue": 980.0},
    {"region": "NA", "product": "beta", "revenue": 300.0},
    {"region": "APAC", "product": "beta", "revenue": 1500.25},
    {"region": "EU", "product": "beta", "revenue": 760.0},
]
COLS = ["region", "product", "revenue"]

session = nb.notebook_session(
    app, question="how does revenue break down by region?"
)
session.register(SALES, columns=COLS, name="sales")
print("session opened over", session.engagement.table)

## Query — cited to the exact source cells

A natural-language question becomes a **read-only-verified** query. The result renders as a
card whose cells are tooltipped with the source cells they rest on, and whose clickable
disclosure lists every cell citation. Nothing in the repr is invented — it can only show
the citations the result actually carries.

In [ ]:
# auto_display is on, so each verb renders its cited artifact inline as a card
result = session.query("total revenue by region")

## Analyze — a bounded, multi-step, cited narrative

The analysis agent runs a bounded plan → query → inspect → refine → synthesize loop, every
step grounded through the query plane. The card shows each finding with the exact cells it
cites and the weakest lineage coverage across the steps.

In [ ]:
analysis = session.analyze("how does revenue break down by region?")

## Chart — content- and data-bound

The cited result becomes a Vega-Lite chart: a C2PA credential binds the rendered bytes
(content-bound) and the figure cites the exact source cells it was built from (data-bound).

In [ ]:
chart = session.chart(session.result, title="Revenue by region")

## Cite and seal the narrative

`cite` assembles the findings into a per-figure data-bound deliverable and closes the
narrative. Sealing mints the signed, audited `DataNarrative` — identical in kind to the one
a script produces — with the **audit id** it was recorded under surfaced inline.

In [ ]:
session.cite(title="Revenue by region")
narrative = session.narrative
print("stages :", " \u2192 ".join(narrative.stage_names))
print("audit id:", narrative.audit_id)
narrative  # renders the chain, the integrity verdict, and the audit id

## Verify — every inline finding re-derives from the bytes

`session.verify()` recomputes the whole chain from the bytes *and* re-executes every query,
analysis, chart, and metric against the content-hashed source — so `data_bound` is true only
when each finding reproduces. The verdict is shown on the session card.

In [ ]:
verdict = session.verify()
print("valid     :", verdict.valid)
print("data-bound:", verdict.data_bound)
print("signed by :", narrative.signed_by)
assert verdict.valid and verdict.data_bound
session  # the session card now shows the data-binding verdict

## Offline and tamper-evident

The sealed narrative round-trips through the wire and still verifies from the bytes alone,
and a **tampered source** is caught — `verify()` flips to not-data-bound even though the
chain itself is intact.

In [ ]:
from vincio import DataNarrative
from vincio.data import DataCatalog, Dataset

restored = DataNarrative.from_wire(narrative.to_wire())
print("round-trips and verifies:", restored.verify(app.contract_signer).valid)

tampered = DataCatalog.of(
    Dataset.from_records([{**r, "revenue": r["revenue"] + 1} for r in SALES], name="sales"),
    name="sales",
)
tamper = session.verify(catalog=tampered)
print("tampered source data-bound:", tamper.data_bound)
assert tamper.data_bound is False and not tamper.valid

## Recap

- `notebook_session(app, ...)` makes an interactive exploration **governed and reproducible
  by construction** — the same signed `DataNarrative` a script produces.
- The reprs show only an artifact's **real, verifiable facts** — clickable cell citations,
  the lineage verdict, the audit id — never a fabricated one.
- `session.verify()` re-derives **every inline finding from the bytes**, and a tampered
  source is caught.

It binds the *existing* plane into the *existing* notebook reprs — an interactive front for
the governed analysis, never a hosted notebook service.